# 04. FE 関数カタログ (`src/03_fe_all.py`)

各モデルが個別に実装していた FE 関数を 1 ファイルに集約したもの。
「あるモデルでは試したが別のモデルでは未適用」という取りこぼしを探すために作られた。

本番で実際に使われるのはモデル別の `src/03_fe_lgbm.py` / `03_fe_xgb.py` / `03_fe_catboost.py` /
`03_fe_realmlp.py`。`03_fe_all.py` は横断比較用のカタログ。

In [ ]:
import os, sys
# リポジトリルートを作業ディレクトリにして、data/ などの相対パスを揃える
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd())

In [ ]:
import inspect
import numpy as np
import pandas as pd
import fe_all

funcs = [(n, f) for n, f in vars(fe_all).items()
         if callable(f) and not n.startswith("_") and getattr(f, "__module__", "") == "fe_all"]
pd.DataFrame(
    [(n, str(inspect.signature(f)), (f.__doc__ or "").strip().split("\n")[0]) for n, f in funcs],
    columns=["関数", "シグネチャ", "説明"]
)

## モデル別 FE ファイルの関数一覧

In [ ]:
import importlib
rows = []
for mod_name in ["fe_lgbm", "fe_xgb", "fe_catboost", "fe_realmlp"]:
    mod = importlib.import_module(mod_name)
    for n, f in vars(mod).items():
        if callable(f) and not n.startswith("_") and getattr(f, "__module__", "") == mod_name:
            rows.append((mod_name, n, (f.__doc__ or "").strip().split("\n")[0][:60]))
pd.DataFrame(rows, columns=["ファイル", "関数", "説明"])

## 動作確認 — 小さなサンプルで出力を見る

実際にどんな列が作られるかを確認する。

In [ ]:
train = pd.read_csv("data/train.csv").head(2000)
test = pd.read_csv("data/test.csv").head(500)
train.head(3)

### digit features — 数値を桁ごとの列にする

年収 84,880 なら 1の位=0、10の位=8、100の位=8… と分解する。
既定のビン数では潰れてしまう細かい値の違いを木に見せる狙い。

**検証結果**: LightGBM / XGBoost では効果なし(+0.0001)、**CatBoost のみ +0.00061**。
CatBoost は既定のビン数が 64 と粗く、年収の値の違いが最も潰れていたため。

In [ ]:
import fe_lgbm
d = fe_lgbm.add_digit_features(train)
print("生成された列数:", d.shape[1])
d.head()

## 効果が確認できた関数・できなかった関数

詳細は `Log.md` の「Feature Engineering 検証結果」表を参照(なぜ試したか / 期待した効果 /
考察まで記録してある)。

**効いたもの**
- 厳密値 Target Encoding(入れ子 CV 版) — 最大の改善要因(+0.003 前後)
- Count / Frequency Encoding — LightGBM・XGBoost で有効、CatBoost では無効
- Triple TE + Smooth Keys — +0.0005〜0.001
- catify(低カーデ数値をカテゴリ扱い) — **CatBoost 固有**(+0.0017)。他モデルでは逆効果

**効かなかったもの**
- 四則演算(diff / ratio / sum / avg) — 3モデルすべてで無効〜悪化
- 交互作用 TE(2列 → 3列 → 6列 → 13列) — すべて無効
- 行フィンガープリント — train 全 668,665 行がユニークなため原理的に機能しない